In [ ]:
# pobranie bibliotek Presidio
%pip install spacy
%pip install presidio_analyzer presidio_anonymizer

# Pseudonimizacja danych PII z użyciem Presidio Anonymizer

Pseudonimizacja to procedura zarządzania danymi i deidentyfikacji, w której pola zawierające dane osobowe w rekordzie są zastępowane jednym lub wieloma sztucznymi identyfikatorami, czyli pseudonimami. (https://en.wikipedia.org/wiki/Pseudonymization)

W tym notebooku pokażemy, jak użyć biblioteki Presidio Anonymizer do pseudonimizacji danych PII. W przykładzie każda wykryta wartość zostanie zastąpiona unikalnym identyfikatorem, na przykład <PERSON_14>. Następnie wykonamy deanonimizację, czyli zamienimy identyfikatory z powrotem na przypisane im wartości PII.

#### **Ważne**: Poniższa logika *nie jest bezpieczna dla wielu wątków* i może dawać niepoprawne wyniki przy równoległym uruchamianiu w środowisku wielowątkowym, ponieważ mapa musi być współdzielona pomiędzy wątkami/procesami.


In [ ]:
from presidio_analyzer import AnalyzerEngine, PatternRecognizer
from presidio_analyzer.nlp_engine import SpacyNlpEngine
from presidio_anonymizer import AnonymizerEngine, DeanonymizeEngine, OperatorConfig
from presidio_anonymizer.operators import Operator, OperatorType

from pathlib import Path
from typing import Dict
from pprint import pprint

#### 1. Użycie `AnalyzerEngine` do wykrywania PII w tekście

In [ ]:
text = "Piotr przekazał swój notes Annie, która później oddała go Zuzannie. Piotr mieszka w Warszawie, a Zuzanna w Krakowie."
print("oryginalny tekst:")
pprint(text)
pl_model_path = Path("pl_blank_model")
if not pl_model_path.exists():
    import spacy
    spacy.blank("pl").to_disk(pl_model_path)
nlp_engine = SpacyNlpEngine(models=[{"lang_code": "pl", "model_name": str(pl_model_path)}])
nlp_engine.load()
analyzer = AnalyzerEngine(nlp_engine=nlp_engine)
person_recognizer = PatternRecognizer(supported_entity="PERSON",
                                      supported_language="pl",
                                      deny_list=["Piotr", "Annie", "Zuzanna", "Zuzannie"])
location_recognizer = PatternRecognizer(supported_entity="LOCATION",
                                        supported_language="pl",
                                        deny_list=["Warszawie", "Krakowie"])
analyzer.registry.add_recognizer(person_recognizer)
analyzer.registry.add_recognizer(location_recognizer)
analyzer_results = analyzer.analyze(text=text, language="pl")
print("wyniki analizatora:")
pprint(analyzer_results)


#### 2. Tworzenie własnego anonimizatora (Operatora), który zastępuje każdą wartość unikalnym identyfikatorem.

Aby utworzyć własny anonimizator, należy zdefiniować klasę dziedziczącą po `Operator` i zaimplementować metodę `operate`. Metoda ta otrzymuje oryginalny tekst oraz słownik `params` z konfiguracją zdefiniowaną przez użytkownika. Powinna zwrócić zanonimizowany tekst.

W tym przykładzie implementujemy także metodę `validate`, aby sprawdzić, czy dostępne są parametry wejściowe, czyli czy zdefiniowano `entity_type` oraz `entity_mapping`, ponieważ są one wymagane przez ten konkretny anonimizator. `entity_mapping` to słownik, który mapuje każdą wartość encji na unikalny identyfikator dla danego typu encji.

In [ ]:
class InstanceCounterAnonymizer(Operator):
    """
    Anonimizator, który zastępuje wartość encji
    licznikiem instancji dla danego typu encji.
    """

    REPLACING_FORMAT = "<{entity_type}_{index}>"

    def operate(self, text: str, params: Dict = None) -> str:
        """Anonimizuje tekst wejściowy."""

        entity_type: str = params["entity_type"]

        # entity_mapping to słownik słowników zawierający mapowania dla każdego typu encji
        entity_mapping: Dict[Dict:str] = params["entity_mapping"]

        entity_mapping_for_type = entity_mapping.get(entity_type)
        if not entity_mapping_for_type:
            new_text = self.REPLACING_FORMAT.format(
                entity_type=entity_type, index=0
            )
            entity_mapping[entity_type] = {}

        else:
            if text in entity_mapping_for_type:
                return entity_mapping_for_type[text]

            previous_index = self._get_last_index(entity_mapping_for_type)
            new_text = self.REPLACING_FORMAT.format(
                entity_type=entity_type, index=previous_index + 1
            )

        entity_mapping[entity_type][text] = new_text
        return new_text

    @staticmethod
    def _get_last_index(entity_mapping_for_type: Dict) -> int:
        """Zwraca ostatni indeks dla danego typu encji."""
        return len(entity_mapping_for_type)

    def validate(self, params: Dict = None) -> None:
        """Sprawdza poprawność parametrów operatora."""

        if "entity_mapping" not in params:
            raise ValueError("Wymagany jest słownik wejściowy o nazwie `entity_mapping`.")
        if "entity_type" not in params:
            raise ValueError("Wymagany jest parametr `entity_type`.")

    def operator_name(self) -> str:
        return "entity_counter"

    def operator_type(self) -> OperatorType:
        return OperatorType.Anonymize

#### 3. Przekazanie nowego operatora do `AnonymizerEngine` i użycie go do anonimizacji tekstu.

In [ ]:
# Utworzenie silnika anonimizacji i dodanie własnego anonimizatora
anonymizer_engine = AnonymizerEngine()
anonymizer_engine.add_anonymizer(InstanceCounterAnonymizer)

# Utworzenie mapowania między typami encji a licznikami
entity_mapping = dict()

# Anonimizacja tekstu

anonymized_result = anonymizer_engine.anonymize(
    text,
    analyzer_results,
    {
        "DEFAULT": OperatorConfig(
            "entity_counter", {"entity_mapping": entity_mapping}
        )
    },
)

print(anonymized_result.text)


Zwróć uwagę, że kolejność jest odwrócona ze względu na sposób zastępowania encji w Presidio.

Ponieważ użytkownik/klient przechowuje `entity_mapping`, można go użyć również do deanonimizacji. Najpierw sprawdźmy jego zawartość.

In [ ]:
pprint(entity_mapping, indent=2)

#### 4. Deanonimizacja tekstu z użyciem `entity_mapping`

Podobnie jak w przypadku operatora anonimizacji, trzeba utworzyć własny operator deanonimizacji. Ten operator zamieni unikalne identyfikatory z powrotem na oryginalne wartości.

In [ ]:
class InstanceCounterDeanonymizer(Operator):
    """
    Deanonimizator, który zastępuje unikalny identyfikator 
    oryginalnym tekstem.
    """

    def operate(self, text: str, params: Dict = None) -> str:
        """Deanonimizuje tekst wejściowy."""

        entity_type: str = params["entity_type"]

        # entity_mapping to słownik słowników zawierający mapowania dla każdego typu encji
        entity_mapping: Dict[Dict:str] = params["entity_mapping"]

        if entity_type not in entity_mapping:
            raise ValueError(f"Nie znaleziono typu encji {entity_type} w mapowaniu encji!")
        if text not in entity_mapping[entity_type].values():
            raise ValueError(f"Nie znaleziono tekstu {text} w mapowaniu encji dla typu {entity_type}!")

        return self._find_key_by_value(entity_mapping[entity_type], text)

    @staticmethod
    def _find_key_by_value(entity_mapping, value):
        for key, val in entity_mapping.items():
            if val == value:
                return key
        return None
    
    def validate(self, params: Dict = None) -> None:
        """Sprawdza poprawność parametrów operatora."""

        if "entity_mapping" not in params:
            raise ValueError("Wymagany jest słownik wejściowy o nazwie `entity_mapping`.")
        if "entity_type" not in params:
            raise ValueError("Wymagany jest parametr `entity_type`.")

    def operator_name(self) -> str:
        return "entity_counter_deanonymizer"

    def operator_type(self) -> OperatorType:
        return OperatorType.Deanonymize


In [ ]:
deanonymizer_engine = DeanonymizeEngine()
deanonymizer_engine.add_deanonymizer(InstanceCounterDeanonymizer)

deanonymized = deanonymizer_engine.deanonymize(
    anonymized_result.text, 
    anonymized_result.items, 
    {"DEFAULT": OperatorConfig("entity_counter_deanonymizer", 
                               params={"entity_mapping": entity_mapping})}
)
print("tekst po anonimizacji:")
pprint(anonymized_result.text)
print("tekst po deanonimizacji:")
pprint(deanonymized.text)